<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="./images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    3.lab.1. Wisconsin Cancer Classification - Neural Networks
  </strong>
</div>

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim

from rich.table import Table
from rich.console import Console

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, Normalizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

Visual configuration of Seaborn for more attractive graphics

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
sns.set_context("notebook", font_scale=1.2)

Rich console instance to display results

In [ ]:
console = Console()

## Data Loading

In [ ]:
breast_cancer = load_breast_cancer(as_frame=True)
X = breast_cancer.data
y = breast_cancer.target

target_names = breast_cancer.target_names

diagnosis_labels = breast_cancer.target.map({0: target_names[0], 1: target_names[1]})

## Preprocessing of data

Split the data with all features into training and testing sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

Method to escalate the data

In [ ]:
def scaler_data(scaler, X_train, X_test):
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled

Method to convert to tensors

In [ ]:
def to_tensors(X_train, y_train, X_test, y_test):
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

    return X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor

Method to train and convert to tensors

In [ ]:
def preprocess_data(scaler, X_train, X_test, y_train, y_test):
    X_train_scaled, X_test_scaled = scaler_data(scaler, X_train, X_test)
    return to_tensors(X_train_scaled, y_train, X_test_scaled, y_test)

### Configurations

Scalers and arhiquitecture to compare in a neural network.

In [ ]:
scalers = {
    "Min-Max": MinMaxScaler(),
    "Standard": StandardScaler(),
    "Mean Normalization": Normalizer(),
}

architectures = {
    "1-Layer (64)": [64],
    "2-Layers (128-64)": [128, 64],
    "2-Layers (128-128)": [128, 128],
    "3-Layers (256-128-128)": [256, 128, 128],
}

### Model Architectures

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size):
        super(NeuralNetwork, self).__init__()

        layers = []
        last_size = input_size

        for h in hidden_sizes:
            layers.append(nn.Linear(last_size, h))
            layers.append(nn.ReLU())
            last_size = h

        layers.append(nn.Linear(last_size, output_size))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

### Training Methods

Method to train a regression model

In [ ]:
def train_model(X_train_scaled, y_train_data, X_test_scaled):

    model = LogisticRegression(penalty="l2", solver="lbfgs", max_iter=1000)
    model.fit(X_train_scaled, y_train_data)

    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

    return y_train_pred, y_test_pred

Method to train a neural network model

In [ ]:
def train_nn_model(model, X_train_tensor, y_train_tensor, epochs, lr):
    loss_history = []

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        y_logits = model(X_train_tensor)
        loss = criterion(y_logits, y_train_tensor)

        loss.backward()
        optimizer.step()

        loss_history.append(loss.item())

    return model, loss_history

### Evaluation Methods

Method to calculate metrics

In [ ]:
def calculate_metrics_for_set(y_test, y_test_pred):
    acc = accuracy_score(y_test, y_test_pred)
    precision = precision_score(y_test, y_test_pred)
    recall = recall_score(y_test, y_test_pred)
    f1 = f1_score(y_test, y_test_pred)

    return acc, precision, recall, f1


def calculate_metrics(
    y_train_model, y_train_model_pred, y_test_model, y_test_model_pred
):

    acc_train, precision_train, recall_train, f1_train = calculate_metrics_for_set(
        y_train_model, y_train_model_pred
    )
    acc_test, precision_test, recall_test, f1_test = calculate_metrics_for_set(
        y_test_model, y_test_model_pred
    )

    return {
        "accuracy_score_train": acc_train,
        "precision_score_train": precision_train,
        "recall_score_train": recall_train,
        "f1_score_train": f1_train,
        "accuracy_score_test": acc_test,
        "precision_score_test": precision_test,
        "recall_score_test": recall_test,
        "f1_score_test": f1_test,
    }

Method to evaluate a neural network model

In [ ]:
def evaluate_nn_model(model, X_train_tensor, X_test_tensor):
    model.eval()
    with torch.no_grad():
        y_train_logits = model(X_train_tensor)
        y_test_logits = model(X_test_tensor)

        y_train_pred = (torch.sigmoid(y_train_logits) >= 0.5).int().numpy()
        y_test_pred = (torch.sigmoid(y_test_logits) >= 0.5).int().numpy()

    return y_train_pred, y_test_pred

Method to train and evaluate network model

In [ ]:
def train_and_evaluate(
    hidden_sizes,
    X_train_tensor,
    y_train_tensor,
    X_test_tensor,
    y_train,
    y_test,
    epochs,
    lr,
):
    torch.manual_seed(42)

    model = NeuralNetwork(X_train_tensor.shape[1], hidden_sizes, 1)
    model, loss_history = train_nn_model(
        model, X_train_tensor, y_train_tensor, epochs, lr
    )

    y_train_pred, y_test_pred = evaluate_nn_model(model, X_train_tensor, X_test_tensor)
    metrics = calculate_metrics(y_train, y_train_pred, y_test, y_test_pred)

    return metrics, loss_history

## Visualization methods

Method to create the comparison table

In [ ]:
def create_comparison_table(title, results):

    comparison_table_normalization = Table(title=title)

    comparison_table_normalization.add_column(
        "Normalization", justify="left", style="cyan"
    )
    comparison_table_normalization.add_column(
        "Model", justify="center", style="green", width=15
    )
    comparison_table_normalization.add_column(
        "Accuracy-\nTrain", justify="right", style="green"
    )
    comparison_table_normalization.add_column(
        "Accuracy-\nTest", justify="right", style="green"
    )
    comparison_table_normalization.add_column(
        "Precision-\nTrain", justify="right", style="green"
    )
    comparison_table_normalization.add_column(
        "Precision-\nTest", justify="right", style="green"
    )
    comparison_table_normalization.add_column(
        "Recall-\nTrain", justify="right", style="green"
    )
    comparison_table_normalization.add_column(
        "Recall-\nTest", justify="right", style="green"
    )
    comparison_table_normalization.add_column(
        "F1-Train", justify="right", style="green"
    )
    comparison_table_normalization.add_column("F1-Test", justify="right", style="green")

    for normalization_type, normalization_metrics in results.items():

        for model_name, metrics in normalization_metrics.items():
            comparison_table_normalization.add_row(
                normalization_type,
                model_name,
                f'{metrics["accuracy_score_train"]:.4f}',
                f'{metrics["accuracy_score_test"]:.4f}',
                f'{metrics["precision_score_train"]:.4f}',
                f'{metrics["precision_score_test"]:.4f}',
                f'{metrics["recall_score_train"]:.4f}',
                f'{metrics["recall_score_test"]:.4f}',
                f'{metrics["f1_score_train"]:.4f}',
                f'{metrics["f1_score_test"]:.4f}',
            )

    console.print(comparison_table_normalization)

Method to show the confusion matrix

In [ ]:
def plot_confusion_matrix(cm, title, ax):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")


def plot_confusion_matrix_comparison(y_test, y_test_pred_log, y_test_pred_nn):
    cm_log = confusion_matrix(y_test, y_test_pred_log)
    cm_nn = confusion_matrix(y_test, y_test_pred_nn)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    plot_confusion_matrix(cm_log, "Confusion Matrix - Logistic Regression", axes[0])
    plot_confusion_matrix(cm_nn, "Confusion Matrix - Neural Network (128,128)", axes[1])

    plt.tight_layout()
    plt.show()

Method to graph loss curves.

In [ ]:
def plot_loss_curves(loss_curves, scalers, architectures):
    n_cols = len(architectures)
    n_rows = len(scalers)

    plt.figure(figsize=(20, 15))

    for i, (label, history) in enumerate(loss_curves.items(), 1):
        plt.subplot(n_rows, n_cols, i)
        plt.plot(history, label="Loss")
        plt.title(label, fontsize=10)
        plt.xlabel("Epochs")
        plt.ylabel("Loss")
        plt.grid(True)

    plt.tight_layout()
    plt.show()

## 1. Create a neural network with different network architectures.

In [ ]:
learning_rate = 0.001
epochs = 500

scaler_metrics = {}
loss_curves = {}

for scaler_name, scaler in scalers.items():
    X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor = preprocess_data(
        scaler, X_train, X_test, y_train, y_test
    )
    scaler_metrics[scaler_name] = {}

    for arch_name, hidden_sizes in architectures.items():
        metrics, loss_history = train_and_evaluate(
            hidden_sizes,
            X_train_tensor,
            y_train_tensor,
            X_test_tensor,
            y_train,
            y_test,
            epochs,
            learning_rate,
        )

        scaler_metrics[scaler_name][arch_name] = metrics
        loss_curves[f"{scaler_name} | {arch_name}"] = loss_history

create_comparison_table(
    "Comparison of metrics between different architectures and normalization methods",
    scaler_metrics,
)

Taking into account the overall metrics, we can say that `the 2-layer (128-128) model with standardization` is the best, as it achieves the best accuracy and F1-Test compared to the others, demonstrating that it is generalizing better. In addition, it maintains perfect performance in training, but without losing capacity in testing, indicating that it achieves a good balance between learning patterns and not overfitting. On the other hand, standardization also helps to keep the variables on the same scale, making it easier for the network to identify relationships more efficiently.

## 2. Compare performance with previous logistic regression results

Since the best performance was achieved with standard normalization, this is what we will use in both logistic regression and neural networks.

In [ ]:
scaler = StandardScaler()
X_train_scaled, X_test_scaled = scaler_data(scaler, X_train, X_test)

comparison_metrics = {}
comparison_metrics["Standard"] = {}

Create the neural network with the best architecture found.

In [ ]:
X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor = to_tensors(
    X_train_scaled, y_train, X_test_scaled, y_test
)

torch.manual_seed(42)
hidden_sizes = [128, 128]
epochs = 500
learning_rate = 0.001

model = NeuralNetwork(X_train_tensor.shape[1], hidden_sizes, 1)
model, loss_history = train_nn_model(
    model, X_train_tensor, y_train_tensor, epochs, learning_rate
)

y_train_pred_nn, y_test_pred_nn = evaluate_nn_model(
    model, X_train_tensor, X_test_tensor
)
metrics = calculate_metrics(y_train, y_train_pred_nn, y_test, y_test_pred_nn)
comparison_metrics["Standard"]["Neural Network"] = metrics

Create the logistic regression model.

In [ ]:
y_train_pred_log, y_test_pred_log = train_model(X_train_scaled, y_train, X_test_scaled)
metrics = calculate_metrics(y_train, y_train_pred_log, y_test, y_test_pred_log)
comparison_metrics["Standard"]["Logistic Regression"] = metrics

Display the results

In [ ]:
create_comparison_table(
    "Comparison of metrics between Neural Network and Logistic Regression with standard normalization",
    comparison_metrics,
)
plot_confusion_matrix_comparison(y_test, y_test_pred_log, y_test_pred_nn)

## 3. Visualize the training process

In [ ]:
plot_loss_curves(loss_curves, scalers, architectures)

## Conclusions

- We can see that there was a slight improvement in the accuracy of the neural network (128-128) compared to logistic regression, as it correctly classified one additional case in the positive class (1). This confirms that, although the change is not very noticeable, the neural network does contribute more.
- It was found that a network without hidden layers is equivalent to logistic regression, reinforcing the idea that the real advantage of networks appears when larger architectures are used.
- In terms of advantages, we can see that the neural network offers greater flexibility to capture nonlinear relationships between variables, which can translate into better performance as the complexity of the problem increases. 
- In terms of disadvantages, we can see that neural networks require more computing resources, more training time, and a greater need for adjustment (such as normalization and architecture choice).
- Logistic regression is much simpler, faster, and easier to interpret, making it practical for problems where the gain in accuracy does not justify the complexity of a neural network.